# Finetuning a Sentence Transformer Model

## Installing Libraries for this Notebook

In [ ]:
!pip install -U transformers
!pip install -U sentence-transformers
!pip install datasets

!pip install evaluate
!pip install seqeval

I would like to note that, for this section, I did not make any changes to the model architecture itself or outside of it.
The SentenceTransformer model was designed specifically to work best with short-form inputs like sentences, passages and paragraphs.

There was also no need to change any hyperparameters of the model itself especially to encode input sentences into embeddings of a fixed-length, since the SentenceTransformer class outputs encodings to a fixed length of 768 encoding values.

More information on the hyperparameters may be found here:
https://www.sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html#id1

# Step 2: Multi-Task Learning Expansion

In this section, I will be doing something a bit differently.

Instead of exclusively using the sBERT sentence_transformers library, I will switch over to using a few libraries, made available by HuggingFace, which will help import the datasets and set-up a sentence transformer model for Sentence Classification and Named Entity Recognition.


### Task A: Sentence Classification


In [ ]:
import os
import json
import numpy as np
import pandas as pd
# Huggingface imports below
import datasets
from datasets import load_dataset, DatasetDict, Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import evaluate
import nltk
from nltk.tokenize import PunktTokenizer
import seaborn
from sklearn.model_selection import train_test_split

In [ ]:
# Importing "IMDb" dataset from Huggingface
imdb_ds = load_dataset("imdb")

# Display the structure of the dataset
print(imdb_ds)

# Display the 'training' split of the dataset
print(imdb_ds["train"])

# Initialize the NLTK Sentence Tokenizer
nltk.download('punkt_tab')
sentence_tokenizer = PunktTokenizer()

# Initialize a Sentence Transformer model
# In this case, I use one of the original models "all-mpnet-base-v2"
# which was trained on more than 1 billion training pairs
# and achieves the highest quantitative results against the other models.
# Those quantitative results may be viewed here:
# https://www.sbert.net/docs/sentence_transformer (link continued below)
# /pretrained_models.html#original-models
sentence_transformer_sbert_model = SentenceTransformer("all-mpnet-base-v2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# The tutorial from HuggingFace, https://huggingface.co/docs/transformers/en/tasks/sequence_classification
# The datasets that get fed during the training loop are first preprocessed
# I will preprocess the dataset in that way
# Then I will preprocess using the dataset as pandas dataframe

In [ ]:
# I convert each dataset subset into pandas DataFrames
# This includes the 'train' subset for training, 'test' subset for testing, and the 'unsupervised' subset in order to find out what it is for.
imdb_train_ds = imdb_ds["train"].to_pandas()
imdb_test_ds = imdb_ds["test"].to_pandas()
imdb_unsupervised_ds = imdb_ds["unsupervised"].to_pandas()

In [ ]:
# Both datasets are 25,000 samples long
print("The size of the training set is:", len(imdb_train_ds))
print("The size of the testing set is:", len(imdb_test_ds))

The size of the training set is: 25000
The size of the testing set is: 25000


In [ ]:
# From our exploratory data analysis notebook
# To Do Item: Include screenshot of barplot from EDA notebook
# I want to split apart the overall texts into sentences, to find out how long every review is
# I will most likely drop the now separated sentences at the end, only needing to use how many sentences there were
# But I will hold onto them for now

# This function will accept an input dataframe
# And return the dataframe with two new columns
# A "sentences" column containing a list of sentences from the overall text column
# And a "num_sentences" column containing the number of sentences in that text.
def tokenize_into_sentences(input_dataframe):
  tokenized_sentences = []
  num_sentences = []

  output_dataframe = input_dataframe

  for entry in input_dataframe['text']:
    result_tokens = sentence_tokenizer.tokenize(entry)

    tokenized_sentences.append(result_tokens)
    num_sentences.append(len(result_tokens))


  output_dataframe["split_sentences"] = tokenized_sentences
  output_dataframe["num_sentences"] = num_sentences

  return output_dataframe

imdb_train_ds = tokenize_into_sentences(imdb_train_ds)
imdb_test_ds = tokenize_into_sentences(imdb_test_ds)

# I take a moment to reorganize the column order of the dataframe, so that label is the final column.
imdb_train_ds = imdb_train_ds.reindex(columns=['text', 'split_sentences', 'num_sentences', 'label'])
imdb_test_ds = imdb_test_ds.reindex(columns=['text', 'split_sentences', 'num_sentences', 'label'])


In [ ]:
print(imdb_train_ds.head())

                                                text  \
0  I rented I AM CURIOUS-YELLOW from my video sto...   
1  "I Am Curious: Yellow" is a risible and preten...   
2  If only to avoid making this type of film in t...   
3  This film was probably inspired by Godard's Ma...   
4  Oh, brother...after hearing about this ridicul...   

                                     split_sentences  num_sentences  label  
0  [I rented I AM CURIOUS-YELLOW from my video st...              9      0  
1  ["I Am Curious: Yellow" is a risible and prete...             11      0  
2  [If only to avoid making this type of film in ...              3      0  
3  [This film was probably inspired by Godard's M...              7      0  
4  [Oh, brother...after hearing about this ridicu...              6      0  


In [ ]:
print(imdb_test_ds.head())

                                                text  \
0  I love sci-fi and am willing to put up with a ...   
1  Worth the entertainment value of a rental, esp...   
2  its a totally average film with a few semi-alr...   
3  STAR RATING: ***** Saturday Night **** Friday ...   
4  First off let me say, If you haven't enjoyed a...   

                                     split_sentences  num_sentences  label  
0  [I love sci-fi and am willing to put up with a...             19      0  
1  [Worth the entertainment value of a rental, es...             11      0  
2  [its a totally average film with a few semi-al...              5      0  
3  [STAR RATING: ***** Saturday Night **** Friday...             11      0  
4  [First off let me say, If you haven't enjoyed ...              7      0  


In [ ]:
# Based on the histogram plots in the other notebook
# To Do Item: Add the screenshot here
# A lot of the reviews are between 0 and 50 sentences
# With a significant amount of reviews that are longer than 50 sentences,
# With some as long as 200 or more sentences

# I will find out a value of num_sentences
# Where below that value, 80% of the data resides
# This is known as the 80th percentile or quantile value
imdb_train_ds_sentences_80_quantile = imdb_train_ds["num_sentences"].quantile(0.8)
imdb_test_ds_sentences_80_quantile = imdb_test_ds["num_sentences"].quantile(0.8)

print(f"The 80th quantile/percentile for num_sentences in imdb_train_ds is: {imdb_train_ds_sentences_80_quantile}")
print(f"The 80th quantile/percentile for num_sentences in imdb_test_ds is: {imdb_test_ds_sentences_80_quantile}")

The 80th quantile/percentile for num_sentences in imdb_train_ds is: 15.0
The 80th quantile/percentile for num_sentences in imdb_test_ds is: 14.0


In [ ]:
# Both the training set and testing set are 25,000 samples long
# When using the free version of Google Colab,
# Training may take a very long time when using CPU only on the entire dataset.
# I saw an estimated completion time of around 170 hours, which is almost 8 days
# In some circumstances, the user may run out of GPU time to train the model.
# Update: A training set of 10,000 samples had an estimation to complete within
#         42 hours, which will cause the notebook session to timeout.

# In that case, I will create two runs.
# One run on the entirety of the provided data
# A second run on a sampled subset of the data

# Also this dataset is provided with only a training set and testing set.
# I will sample a portion of the training set, to use as a validation set.
# Which means I will take 5000 samples from the training set for the validation
# set.

# For the smaller equivalent datasets intended for "CPU only" training,
# I will first sample the training set down to 5,000 samples,
# Then I will take 1,000 samples from that training set for the validation set.
# I will sample the test set down to 2,000 samples
# Update: At this distribution, the training time for 3
# training epochs or iterations is expected to be 16 hours

# To maintain a "reasonable" CPU only training time,
# I will first sample the training set down to 3,000 samples,
# Then I will take 500 samples from that training set for the validation set.
# I will sample the test set down to 2,000 samples
# Update: At this distrubtion, the training time for 3
# training epochs or iterations is expected to be 5 hours.

is_gpu_connected = False
notebook_environment_values_dictionary = dict(os.environ)
notebook_environment_values_dictionary_json_string = json.dumps(notebook_environment_values_dictionary, indent=4)
# print(notebook_environment_values_dictionary_json_string)

if notebook_environment_values_dictionary["COLAB_GPU"] == "" and notebook_environment_values_dictionary["COLAB_TPU_1VM"] == "":
  # This condition means that we are in a "CPU only" runtime
  # So I will set a flag that the data will need to be sampled down
  # So that training times will be more "reasonable"
  is_gpu_connected = False
elif notebook_environment_values_dictionary["COLAB_GPU"] != "" or notebook_environment_values_dictionary["COLAB_TPU_1VM"] != "":
  # This condition means that a GPU or TPU accelerator is connected
  # So I will set a flag that the data will not need to be sampled down
  is_gpu_connected = True
else:
  # This is the default condition
  # Where the data will need to be sampled down
  is_gpu_connected = False

if is_gpu_connected == False:
  imdb_train_ds = imdb_train_ds.sample(n=1000)
  imdb_validation_ds = imdb_train_ds.sample(n=200)
  imdb_train_ds = imdb_train_ds.drop(imdb_validation_ds.index)

  imdb_test_ds = imdb_test_ds.sample(2000)
else: # This condition is when is_gpu_connected is True
  imdb_validation_ds = imdb_train_ds.sample(n=5000)
  imdb_train_ds = imdb_train_ds.drop(imdb_validation_ds.index)

  imdb_test_ds = imdb_test_ds.sample(20000)

In [ ]:
# Preprocess the data
# We also establish the label mappings to their id values
id2label = {0: "Negative Review", 1: "Positive Review"}
label2id = {"Negative Review": 0, "Positive Review": 1}

imdb_train_ds['review_value'] = imdb_train_ds['label'].replace(id2label)
imdb_test_ds['review_value'] = imdb_test_ds['label'].replace(id2label)

In [ ]:
print(imdb_train_ds)

                                                    text  label  \
20980  This was one of those films I probably never w...      1   
21325  I remember this show from Swedish television. ...      1   
2323   This movie is yet another in the long line of ...      0   
4219   A bit of a disappointing film, I'd say: the ac...      0   
15039  It's a good show, and I find it funny. Finally...      1   
...                                                  ...    ...   
7440   its awful i cant believe that one of the great...      0   
2586   German nut case Jörg Buttgereit apparently has...      0   
15329  The film is side spliting from the outset, Edd...      1   
21887  One of my favorite movies to date starts as an...      1   
23704  The title has many meanings - the boxing ring,...      1   

          review_value  
20980  Positive Review  
21325  Positive Review  
2323   Negative Review  
4219   Negative Review  
15039  Positive Review  
...                ...  
7440   Negative Revi

In [ ]:
print(imdb_test_ds)

                                                    text  label  \
12525  Although Bullet In The Brain is, without quest...      1   
1423   Being a HUGE fan of the bottom series i was re...      0   
1778   This movie over does it on the cgi i mean sci-...      0   
7962   I will repeat - what a stupid scenario.<br /><...      0   
35     this film has no plot, no good acting, to be h...      0   
...                                                  ...    ...   
20347  Wealthy businessman Bill Compton (played by De...      1   
7870   One of the worst movies I've ever seen. When I...      0   
7436   Contrary to most of the comments in this secti...      0   
13415  Can you capture the moment? When first you hea...      1   
17234  All Boris Karloff fans will love this classic ...      1   

          review_value  
12525  Positive Review  
1423   Negative Review  
1778   Negative Review  
7962   Negative Review  
35     Negative Review  
...                ...  
20347  Positive Revi

In [ ]:
## We load an accuracy evaluation metric for our compute_metrics method
accuracy = evaluate.load("accuracy")
# We load a tokenizer and establish a preprocessing_function
# sentence_transfomer_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2", low_cpu_mem_usage=True)
sentence_transfomer_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

def to_datasets_Dataset_object(input_data):
  dataset_object = Dataset.from_pandas(input_data)

  return dataset_object

def preprocess_text_into_tokens_function(input_data, column_to_tokenize):
  output_tokens = []
  output_input_ids = []
  output_attention_masks = []

  output_data = input_data

  for i in range(len(input_data)):
    tokens = sentence_transfomer_tokenizer.tokenize(input_data.iloc[i][column_to_tokenize], truncation=True)
    tokenizer_outputs = sentence_transfomer_tokenizer(input_data.iloc[i][column_to_tokenize], truncation=True)

    #print(tokenizer_outputs)

    output_tokens.append(tokens)
    output_input_ids.append(tokenizer_outputs["input_ids"])
    output_attention_masks.append(tokenizer_outputs["attention_mask"])

  output_data["tokens"] = output_tokens
  output_data["input_ids"] = output_input_ids
  output_data["attention_masks"] = output_attention_masks

  return output_data


In [ ]:
# I will preform two preprocessing
# One that will get tokens, input_ids, and attention_masks from a DataFrame
# One from the Huggingface tutorial that gets the same values, from a datasets.Dataset value

# Here is using our own preprocess function
# This is mostly to do our own data analysis later
# imdb_processed_train_ds = preprocess_text_into_tokens_function(imdb_train_ds, column_to_tokenize="text")
# imdb_processed_test_ds = preprocess_text_into_tokens_function(imdb_test_dsm column_to_tokenize="text")

# Here is the method from the tutorial
def preprocess_function(examples):
    return sentence_transfomer_tokenizer(examples["text"], truncation=True)

# Converting from pandas.DataFrame to datasets.Dataset
train_datasets_object = to_datasets_Dataset_object(imdb_train_ds)
test_datasets_object = to_datasets_Dataset_object(imdb_test_ds)

tokenized_imdb_train_ds = train_datasets_object.map(preprocess_function, batched=True)
tokenized_imdb_test_ds = test_datasets_object.map(preprocess_function, batched=True)

print(tokenized_imdb_train_ds[0])


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'text': 'This was one of those films I probably never would have picked off the shelf , but it came on IFC one day and I said - Eric Stolz, William Forsythe...why not? If I\'d changed the channel, I would have really missed a treasure. <br /><br />The subject is depressing - young author paralyzed in climbing accident convalesces in lower-class rehabilitation center. It would have been so easy and tempting to make this a manipulative tear-jerker. But, that doesn\'t happen because it was written by Neal Jimenez, after he himself was accidently paralyzed. No Hollywood happiness here. All of the patients in the ward come from wildly different backgrounds, but they share a feeling of helplessness, of being at the mercy of others. Stolz is very good as a "lone wolf" type, forced into embarrassing dependence on his girlfriend (Helen Hunt); Wesley Snipes is fine as a former ladies\' man whose family is falling apart; but William Forsythe takes the cake as a tough guy determined to make someo

In [ ]:
# We load a pretrained sentence transformer model
# imbd_sentence_classifier_model = AutoModelForSequenceClassification.from_pretrained("sentence-transformers/all-mpnet-base-v2", num_labels=2, id2label=id2label, label2id=label2id, low_cpu_mem_usage=True)
imbd_sentence_classifier_model = AutoModelForSequenceClassification.from_pretrained("sentence-transformers/all-mpnet-base-v2", num_labels=2, id2label=id2label, label2id=label2id)
# And a data collator for the trainer
data_collator = DataCollatorWithPadding(tokenizer=sentence_transfomer_tokenizer)

# Pass in a dataset object that only has "attention_mask," "input_ids," and "label" as values
# So we remove the text column
tokenized_imdb_train_ds = tokenized_imdb_train_ds.remove_columns(['text'])
tokenized_imdb_test_ds = tokenized_imdb_test_ds.remove_columns(['text'])

# Set the format to pytorch tensors
tokenized_imdb_train_ds.set_format('torch')

# I noticed that the notebook crashes when the batch size is 16 and
# that the notebook runtype is CPU only
# So I will change the batch size to 8 when it is on CPU only

if is_gpu_connected == False:
  batch_size = 8
else:
  batch_size = 16 # This number can be experimented with

# Establishing trainer
training_args = TrainingArguments(
    output_dir="NA_imdb_trained_sentence_transformer",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    use_cpu=False,
    report_to=["none"]
)

trainer = Trainer(
    model=imbd_sentence_classifier_model,
    args=training_args,
    train_dataset=tokenized_imdb_train_ds,
    eval_dataset=tokenized_imdb_test_ds,
    tokenizer=sentence_transfomer_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-14-d85eb0784fcd>:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


In [ ]:
text = "The original Korean version of this movie was allegedly superior to the American one. I haven't seen either of them, so I can't comment on it, but a lot of people nowadays say they vastly prefer the American version."

In [ ]:
classifier = pipeline("sentiment-analysis", model="NA_imbd_trained_sentence_transformer/checkpoint-782")
classifier(text)